# NLP1 · Project 1 — Occupation classification on *Bias in Bios*

**Task.** 28-way occupation classification from `hard_text` (bio with the first, title-revealing sentence removed).
Gender (`0 = male`, `1 = female`) is **never a feature** — it is only used to evaluate *for whom* the classifier works.

**Design (3 × 3):**

| | Log-linear (A1) | 1-hidden-layer MLP (A2) | Deeper MLP (A3, mine) |
|---|---|---|---|
| Token indicators (R1) | ✓ | ✓ | ✓ |
| Bag-of-words counts (R2) | ✓ | ✓ | ✓ |
| TF-IDF (R3, mine) | ✓ | ✓ | ✓ |

**Evaluation.** k-fold CV on a stratified pool from `train` · primary metric **macro-F1** (accuracy reported too) · fairness: per-occupation **TPR gender gap** + **RMS gap** · Bayesian correlated-t with a ROPE.
`dev` and `test` stay untouched (reserved for later projects / a devtest role).

**Notebook map:** 0 Setup · 1 Data & EDA · 2 Splits · 3 Tokenizer · 4 Representations · 5 Architectures · 6 Metrics · 7 CV runner · 8 Results · 9 Bayesian comparison · 10 Fairness analysis · 11 Extra · 12 Slide summary · 13 AI-assistance log

## 0 · Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, json, pickle, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from scipy.sparse import csr_matrix, vstack as sp_vstack
from scipy.stats import pearsonr
from datasets import load_dataset
from tokenizers import Tokenizer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)

warnings.simplefilter(action="ignore", category=FutureWarning)

# notebook lives next to the tutorial (root/part1), same import trick as the tutorial
root = Path.cwd().parent
sys.path.insert(0, str(root))
import part1

In [ ]:
# ---- experiment configuration (everything you might be asked to justify lives here) ----
SEEDS      = [0]          # more seeds (e.g. [0, 1, 2]) -> more observations -> tighter posteriors
K          = 5            # CV folds
SUBSAMPLE  = 100_000      # size of the CV pool drawn from train; None = all ~257k (slow for MLPs)
VOCAB_SIZE = 5_000        # BPE vocabulary size
MIN_FREQ   = 2            # BPE min merge frequency
PRIMARY    = "macro_f1"   # metric used for the Bayesian comparisons
ROPE       = 0.01         # ROPE half-width for PRIMARY -> justify in §9
ROPE_GAP   = 0.01         # ROPE for the RMS TPR gap
MIN_CELL   = 10           # min #examples in an (occupation, gender) cell to compute its TPR
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"   # sparse ops are not reliable on MPS
RERUN      = False        # False = load cached fold results from disk if present

RESULTS = Path("results/project1"); RESULTS.mkdir(parents=True, exist_ok=True)
np.random.seed(0); torch.manual_seed(0)

## 1 · Data & exploratory analysis

In [ ]:
ds = {s: load_dataset("LabHC/bias_in_bios", split=s).to_pandas() for s in ["train", "dev", "test"]}
for s, df in ds.items():
    print(f"{s:5s} {df.shape}")
train_df = ds["train"]

NAMES = ["accountant","architect","attorney","chiropractor","comedian","composer",
         "dentist","dietitian","dj","filmmaker","interior_designer","journalist",
         "model","nurse","painter","paralegal","pastor","personal_trainer",
         "photographer","physician","poet","professor","psychologist","rapper",
         "software_engineer","surgeon","teacher","yoga_teacher"]
GENDER = {0: "male", 1: "female"}
outcome_space = part1.clslib.NamedClasses(NAMES)
train_df.head()

### 1.1 Label distribution & trivial baselines

In [ ]:
prop = train_df.profession.map(dict(enumerate(NAMES))).value_counts(normalize=True).sort_values()
prop.plot.barh(figsize=(6, 8)); plt.xlabel("share of train"); plt.show()

y_all = train_df.profession.to_numpy()
maj = np.full_like(y_all, np.bincount(y_all).argmax())
print(f"majority-class baseline: accuracy = {accuracy_score(y_all, maj):.3f}, "
      f"macro-F1 = {f1_score(y_all, maj, average='macro'):.3f}")

### 1.2 Gender × occupation

In [ ]:
ct = pd.crosstab(train_df.profession.map(dict(enumerate(NAMES))), train_df.gender.map(GENDER))
ct["n"] = ct.sum(axis=1)
ct["pct_female"] = ct["female"] / ct["n"]
ct = ct.sort_values("pct_female")
PI_FEMALE = ct["pct_female"]          # reused in §10
display(ct)

ct["pct_female"].plot.barh(figsize=(6, 8)); plt.axvline(0.5, ls="--", c="k")
plt.xlabel("% female (train)"); plt.show()

print("smallest (occupation, gender) cells:")
display(ct[["male", "female"]].stack().sort_values().head(8))

### 1.3 Text: length, examples, templates, noise

In [ ]:
train_df["n_chars"] = train_df.hard_text.str.len()
train_df["n_words"] = train_df.hard_text.str.split().str.len()
display(train_df[["n_chars", "n_words"]].describe().round(1))

for _, r in train_df.sample(5, random_state=1).iterrows():
    print(f"[{NAMES[r.profession]} | {GENDER[r.gender]}] {r.hard_text}\n")

In [ ]:
# how common are gender words, templates, label leaks and personal data?
patterns = {
    "explicit pronoun/title": r"\b(?:he|she|his|her|him|mr|mrs|ms)\b",
    "NPI template":           r"NPI Number",
    "'Having more than N years' template": r"Having more than \d+ years",
    "ALL-CAPS phrase (label leak?)":       r"\b[A-Z]{4,} [A-Z]{4,}\b",
    "e-mail":                 r"\S+@\S+\.\w+",
    "phone number":           r"\(?\d{3}\)?[ -]\d{3}-\d{4}",
}
for name, pat in patterns.items():
    case_sensitive = name != "explicit pronoun/title"
    share = train_df.hard_text.str.contains(pat, case=case_sensitive, regex=True).mean()
    print(f"{name:40s} {share:6.2%}")

**Observations (fill in → *Data & metrics* slide):**
- imbalance: …
- most skewed occupations: …
- templates / leaks / noise: …
- why accuracy alone is not enough here: …

## 2 · Experimental splits

- CV pool = stratified subsample of `train` (size `SUBSAMPLE`); `dev`/`test` untouched.
- Folds stratified by **occupation × gender**, so rare cells (e.g. male paralegals) appear in every fold.
- **The same folds are used for every system** — the Bayesian model analyses *paired* fold-wise differences.
- Gender stays aligned with every example through the index arrays (the tutorial helper only keeps `(text, label)`).

In [ ]:
strata_all = train_df.profession * 2 + train_df.gender
pool = (train_df if SUBSAMPLE is None else
        train_test_split(train_df, train_size=SUBSAMPLE, stratify=strata_all, random_state=0)[0])
pool = pool.reset_index(drop=True)

TEXTS = pool.hard_text.tolist()
Y     = pool.profession.to_numpy()
G     = pool.gender.to_numpy()
strata = Y * 2 + G

FOLDS = {seed: list(StratifiedKFold(K, shuffle=True, random_state=seed).split(TEXTS, strata))
         for seed in SEEDS}
json.dump({s: [[tr.tolist(), va.tolist()] for tr, va in f] for s, f in FOLDS.items()},
          open(RESULTS / "folds.json", "w"))                       # reproducibility

def pairs(idx, texts=None):
    texts = TEXTS if texts is None else texts
    return [(texts[i], int(Y[i])) for i in idx]

print(len(TEXTS), "bios in the CV pool;", [len(va) for _, va in FOLDS[SEEDS[0]]], "per validation fold")

## 3 · Tokenizer

Decisions to justify: **training data** (full `train` split — big enough, in-domain; BPE never sees labels, so fitting it on all of `train` is a negligible leak into the CV folds), **vocab size**, **min frequency**, **normalisation**.

In [ ]:
tok_path = RESULTS / f"bpe_vs{VOCAB_SIZE}_mf{MIN_FREQ}.json"
if tok_path.exists() and not RERUN:
    tokenizer = Tokenizer.from_file(str(tok_path))
else:
    tokenizer = part1.experiment.train_tokenizer(train_df.hard_text.tolist(), VOCAB_SIZE, MIN_FREQ)
    tokenizer.save(str(tok_path))
V = tokenizer.get_vocab_size()
print("vocab size:", V)

In [ ]:
# what do units look like? are occupation words fragmented? are pronouns single tokens?
print(tokenizer.encode(TEXTS[0]).tokens[:40])
for w in ["he", "she", "his", "her", "chiropractor", "paralegal", "dermatology", "choreographer", "attorney"]:
    print(f"{w:15s} -> {tokenizer.encode(w).tokens}")

In [ ]:
# cache token ids once (encoders below look texts up here instead of re-tokenising every fold)
IDS = {t: np.array(e.ids, dtype=np.int64) for t, e in zip(TEXTS, tokenizer.encode_batch(TEXTS))}
lens = np.array([len(v) for v in IDS.values()])
print("BPE tokens per bio (5/50/95 pct):", np.percentile(lens, [5, 50, 95]))

## 4 · Representations

Same interface as the tutorial's `Encoder`: `len(enc) = D`, `enc(text) -> np.array[D]`.

| | Coordinate $h_v(x)$ | Captures | Ignores |
|---|---|---|---|
| R1 indicator | $\mathbb 1[v \in x]$ | which units occur | frequency, order, length |
| R2 BoW | $\text{count}_v(x)$ | + repetition | order; length-sensitive |
| R3 TF-IDF | $\log(1+\text{count}_v)\cdot\text{idf}_v$, L2-normalised | distinctive units; length-normalised | order |

**Why TF-IDF as R3 (fill in):** …

In [ ]:
class Encoder:
    def __init__(self):
        self.V = V
    def __len__(self):
        return self.V
    def token_ids(self, x):
        return IDS[x] if x in IDS else np.array(tokenizer.encode(x).ids, dtype=np.int64)

class TokenIndicator(Encoder):                     # R1
    def __call__(self, x):
        v = np.zeros(self.V); v[np.unique(self.token_ids(x))] = 1.0
        return v

class BagOfWords(Encoder):                         # R2
    def __call__(self, x):
        return np.bincount(self.token_ids(x), minlength=self.V).astype(float)

class TfIdf(Encoder):                              # R3 — idf fitted on the TRAINING fold only
    def fit(self, texts):
        df = np.zeros(self.V)
        for t in texts:
            df[np.unique(self.token_ids(t))] += 1
        self.idf = np.log((1 + len(texts)) / (1 + df)) + 1
        return self
    def __call__(self, x):
        v = np.log1p(np.bincount(self.token_ids(x), minlength=self.V)) * self.idf
        n = np.linalg.norm(v)
        return v / n if n > 0 else v

# factory: receives the training-fold texts (only TF-IDF needs them)
REPRESENTATIONS = {
    "indicator": lambda train_texts: TokenIndicator(),
    "bow":       lambda train_texts: BagOfWords(),
    "tfidf":     lambda train_texts: TfIdf().fit(train_texts),
}

In [ ]:
# sanity check on one bio
x = TEXTS[0]
for name, make in REPRESENTATIONS.items():
    h = make(TEXTS[:2000])(x)
    print(f"{name:10s} D={len(h)}  nonzeros={int((h > 0).sum())}  sum={h.sum():.2f}  max={h.max():.2f}")

## 5 · Architectures

| | Model | Notes |
|---|---|---|
| A1 | log-linear softmax regression (`part1.loglin`) | convex, interpretable weights |
| A2 | MLP, 1 hidden layer + ReLU (`part1.smlp`) | first layer on sparse BoW = sum of learned token embeddings |
| A3 | **deeper MLP** (2 hidden layers, more dropout) — own code | does extra non-linearity help with 28 overlapping classes? |

Epoch selection for A2/A3 uses the fold's validation split (as in the tutorial) → slightly optimistic, but identical protocol for all neural systems.
All functions share one signature: `(train_pairs, val_pairs, encoder, seed) -> predicted labels for val_pairs`.

In [ ]:
LOGLIN_ARGS = dict()   # check `part1.loglin.fit?` for what it accepts (max_iter, C, ...)

def fit_predict_loglin(train_pairs, val_pairs, enc, seed):
    model = part1.loglin.fit(train_pairs, enc, save_to=None, **LOGLIN_ARGS)
    return np.asarray(part1.loglin.predict([t for t, _ in val_pairs], model, enc))

In [ ]:
MLP_ARGS = dict(hidden_dim=128, dropout=0.1, batch_size=64, learning_rate=1e-4,
                weight_decay=1e-4, num_epochs=10)   # tutorial defaults; larger batch + lr is faster

def fit_predict_mlp(train_pairs, val_pairs, enc, seed):
    torch.manual_seed(seed)
    _, best_epoch, best_preds = part1.smlp.train_neural_network_model(
        train_pairs, val_pairs, num_classes=len(NAMES), encoder=enc,
        save_to=str(RESULTS / "tmp_mlp"), device=DEVICE, **MLP_ARGS)
    return np.asarray(best_preds)   # verify: val predictions at best epoch, in val_pairs order

In [ ]:
DEEP_ARGS = dict(hidden=(256, 128), dropout=0.3, lr=1e-3, weight_decay=1e-4, batch_size=256, epochs=8)

class DeepMLP(nn.Module):
    def __init__(self, D, K, hidden=(256, 128), dropout=0.3):
        super().__init__()
        self.inp = nn.Linear(D, hidden[0])          # applied to a sparse input
        layers, d = [nn.ReLU(), nn.Dropout(dropout)], hidden[0]
        for h in hidden[1:]:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]; d = h
        self.body = nn.Sequential(*layers, nn.Linear(d, K))   # logits (CrossEntropyLoss = log-softmax + NLL)
    def forward(self, x):
        h = torch.sparse.mm(x, self.inp.weight.t()) + self.inp.bias
        return self.body(h)

def encode_matrix(texts, enc):
    return sp_vstack([csr_matrix(enc(t)) for t in texts]).tocsr()

def to_torch(X):
    X = X.tocoo()
    return torch.sparse_coo_tensor(np.vstack([X.row, X.col]), X.data, X.shape, dtype=torch.float32)

@torch.no_grad()
def predict_deep(model, X, bs=2048):
    model.eval()
    return np.concatenate([model(to_torch(X[i:i + bs]).to(DEVICE)).argmax(1).cpu().numpy()
                           for i in range(0, X.shape[0], bs)])

def fit_predict_deep(train_pairs, val_pairs, enc, seed, hidden, dropout, lr, weight_decay, batch_size, epochs):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    Xtr = encode_matrix([t for t, _ in train_pairs], enc); ytr = torch.tensor([y for _, y in train_pairs])
    Xva = encode_matrix([t for t, _ in val_pairs], enc);   yva = np.array([y for _, y in val_pairs])
    model = DeepMLP(Xtr.shape[1], len(NAMES), hidden, dropout).to(DEVICE)
    opt, loss_fn = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay), nn.CrossEntropyLoss()
    best, best_preds = -1.0, None
    for ep in range(epochs):
        model.train()
        for b in np.array_split(rng.permutation(Xtr.shape[0]), max(1, Xtr.shape[0] // batch_size)):
            loss = loss_fn(model(to_torch(Xtr[b]).to(DEVICE)), ytr[torch.as_tensor(b)].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step()
        preds = predict_deep(model, Xva)
        score = f1_score(yva, preds, average="macro")
        print(f"   epoch {ep}: val macro-F1 = {score:.4f}")
        if score > best:
            best, best_preds = score, preds
    return best_preds

ARCHITECTURES = {
    "loglin": fit_predict_loglin,
    "mlp":    fit_predict_mlp,
    "deep":   lambda tr, va, enc, seed: fit_predict_deep(tr, va, enc, seed, **DEEP_ARGS),
}

## 6 · Metrics

- **accuracy** (comparability with literature), **macro-F1** (primary: every occupation counts), **balanced accuracy** (= mean per-class TPR)
- **TPR gender gap** per occupation: $\mathrm{Gap}_y = \mathrm{TPR}_{F,y} - \mathrm{TPR}_{M,y}$, skipped if a cell has `< MIN_CELL` examples
- **RMS gap** $= \sqrt{\tfrac{1}{|C|}\sum_y \mathrm{Gap}_y^2}$ (lower = fairer)

In [ ]:
def evaluate(y, yhat, g):
    out = dict(acc=accuracy_score(y, yhat),
               macro_f1=f1_score(y, yhat, average="macro"),
               bal_acc=balanced_accuracy_score(y, yhat))
    gaps = {}
    for c, name in enumerate(NAMES):
        tpr = {}
        for gv in (0, 1):
            m = (y == c) & (g == gv)
            tpr[gv] = (yhat[m] == c).mean() if m.sum() >= MIN_CELL else np.nan
        gaps[name] = tpr[1] - tpr[0]                    # female − male
    valid = np.array([v for v in gaps.values() if not np.isnan(v)])
    out["rms_gap"] = float(np.sqrt(np.mean(valid ** 2)))
    out["gaps"] = gaps
    return out

## 7 · Cross-validation runner

One function runs a (representation, architecture) pair over all seeds × folds, stores predictions + metrics, and caches to disk (`RERUN=False` reloads).
Full grid = 9 systems. **If short on time**, run an L-shape first: all 3 representations with `loglin` (cheap), then all 3 architectures with the best representation.

In [ ]:
def run_system(rep, arch, texts=None, tag=""):
    texts = TEXTS if texts is None else texts
    name = f"{arch}_{rep}{tag}"
    path = RESULTS / f"{name}.pkl"
    if path.exists() and not RERUN:
        return pickle.load(open(path, "rb"))
    runs = []
    for seed in SEEDS:
        for k, (tr, va) in enumerate(FOLDS[seed]):
            enc = REPRESENTATIONS[rep]([texts[i] for i in tr])
            yhat = ARCHITECTURES[arch](pairs(tr, texts), pairs(va, texts), enc, seed)
            m = evaluate(Y[va], yhat, G[va])
            runs.append(dict(seed=seed, fold=k, val_idx=va, y_pred=yhat, **m))
            print(f"{name:18s} seed={seed} fold={k}  acc={m['acc']:.4f}  "
                  f"macroF1={m['macro_f1']:.4f}  rmsGap={m['rms_gap']:.4f}")
    pickle.dump(runs, open(path, "wb"))
    return runs

In [ ]:
%%time
SYSTEMS = [(r, a) for a in ARCHITECTURES for r in REPRESENTATIONS]   # 9 systems
RESULTS_BY = {}
for rep, arch in SYSTEMS:
    RESULTS_BY[f"{arch}_{rep}"] = run_system(rep, arch)

## 8 · Results

In [ ]:
scores = pd.DataFrame([dict(system=s, seed=r["seed"], fold=r["fold"],
                            acc=r["acc"], macro_f1=r["macro_f1"], bal_acc=r["bal_acc"], rms_gap=r["rms_gap"])
                       for s, runs in RESULTS_BY.items() for r in runs])
summary = scores.groupby("system")[["acc", "macro_f1", "bal_acc", "rms_gap"]].agg(["mean", "std"]).round(4)
display(summary.sort_values(("macro_f1", "mean"), ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, met in zip(axes, ["acc", "macro_f1", "rms_gap"]):
    scores.boxplot(column=met, by="system", ax=ax, rot=60); ax.set_title(met); ax.set_xlabel("")
plt.suptitle(""); plt.tight_layout(); plt.show()

In [ ]:
# per-class view of the best system (all folds of the first seed pooled)
BEST = scores.groupby("system")[PRIMARY].mean().idxmax()
runs0 = [r for r in RESULTS_BY[BEST] if r["seed"] == SEEDS[0]]
yt = np.concatenate([Y[r["val_idx"]] for r in runs0])
gt = np.concatenate([G[r["val_idx"]] for r in runs0])
yp = np.concatenate([r["y_pred"] for r in runs0])
print("best system:", BEST)
print(classification_report(yt, yp, target_names=NAMES, digits=3))

cm = confusion_matrix(yt, yp, normalize="true")
top = sorted(((NAMES[i], NAMES[j], cm[i, j]) for i in range(len(NAMES)) for j in range(len(NAMES)) if i != j),
             key=lambda t: -t[2])[:15]
display(pd.DataFrame(top, columns=["true", "predicted", "rate"]).round(3))

## 9 · Bayesian comparison (Benavoli correlated t)

**ROPE justification (fill in):** units of macro-F1; why `ROPE` = …; how it relates to fold-to-fold std; what difference a user would notice.
Comparisons hold one factor fixed: representation effects within each architecture, architecture effects within each representation.
Check the sign convention of `part1.analysis.BenavoliCorrelatedT` (which system "wins" when the difference is positive). For `rms_gap`, *lower* is better.

In [ ]:
def fold_scores(system, metric=PRIMARY):
    runs = sorted(RESULTS_BY[system], key=lambda r: (r["seed"], r["fold"]))   # same order for every system
    return np.array([r[metric] for r in runs])

def compare(a, b, metric=PRIMARY, rope=ROPE, plot=False):
    post = part1.analysis.BenavoliCorrelatedT(fold_scores(a, metric), fold_scores(b, metric), names=[a, b])
    p_a, p_tie, p_b = post.probs(rope)
    if plot:
        post.plot(rope)
    return dict(A=a, B=b, metric=metric, rope=rope, P_A=p_a, P_tie=p_tie, P_B=p_b, effect=post.effect_size)

reps, archs = list(REPRESENTATIONS), list(ARCHITECTURES)
PAIRS  = [(f"{a}_{r1}", f"{a}_{r2}") for a in archs for i, r1 in enumerate(reps) for r2 in reps[i + 1:]]
PAIRS += [(f"{a1}_{r}", f"{a2}_{r}") for r in reps for i, a1 in enumerate(archs) for a2 in archs[i + 1:]]
PAIRS  = [(a, b) for a, b in PAIRS if a in RESULTS_BY and b in RESULTS_BY]

bayes = pd.DataFrame([compare(a, b) for a, b in PAIRS])
display(bayes.round(3))

In [ ]:
# posterior plots for the 2–3 comparisons that go on the Results slide
for a, b in PAIRS[:3]:          # pick the interesting ones
    compare(a, b, plot=True); plt.show()

# the same machinery on the fairness metric
display(pd.DataFrame([compare(a, b, metric="rms_gap", rope=ROPE_GAP) for a, b in PAIRS]).round(3))

## 10 · Fairness analysis

Does the TPR gap track how female an occupation already is (compounding imbalance)? Which gendered confusions drive it?

In [ ]:
def gap_table(system):
    g = pd.DataFrame([r["gaps"] for r in RESULTS_BY[system]]).mean()      # mean over folds
    return pd.DataFrame({"gap_F_minus_M": g, "pct_female": PI_FEMALE.reindex(g.index)}).dropna()

def plot_gaps(system, ax):
    t = gap_table(system)
    r, _ = pearsonr(t.pct_female, t.gap_F_minus_M)
    ax.scatter(t.pct_female, t.gap_F_minus_M, s=15)
    for n, row in t.iterrows():
        ax.annotate(n, (row.pct_female, row.gap_F_minus_M), fontsize=7)
    ax.axhline(0, c="k", lw=.5); ax.axvline(.5, c="k", lw=.5, ls="--")
    ax.set(title=f"{system} (r = {r:.2f})", xlabel="% female in occupation", ylabel="TPR gap (F − M)")

show = [BEST, "loglin_indicator", "mlp_indicator"]
fig, axes = plt.subplots(1, len(show), figsize=(6 * len(show), 5))
for ax, s in zip(np.atleast_1d(axes), show):
    plot_gaps(s, ax)
plt.tight_layout(); plt.show()

In [ ]:
# where do women vs men of the same occupation get misclassified? (best system)
for occ in ["surgeon", "nurse", "paralegal", "software_engineer", "interior_designer", "rapper"]:
    c = NAMES.index(occ)
    for gv in (0, 1):
        m = (yt == c) & (gt == gv)
        dist = pd.Series(yp[m]).map(dict(enumerate(NAMES))).value_counts(normalize=True).head(3).round(3)
        print(f"{occ:18s} {GENDER[gv]:6s} n={m.sum():5d}  ->  {dist.to_dict()}")

## 11 · Extra (optional) — explicit gender indicators

1. **Scrubbing:** rerun the best system with pronouns/titles removed. Does accuracy change? Does the RMS gap drop? (Names cannot be removed reliably here — the first sentence with the full name is already gone.)
2. **Swap test:** flip he↔she etc. in validation bios and count how many predictions change.
3. **Weights:** which tokens does the log-linear model associate with each occupation — do pronouns appear?

In [ ]:
GENDER_WORDS = r"\b(?:he|she|his|her|hers|him|himself|herself|mr|mrs|ms)\b\.?"
def scrub(t):
    return re.sub(GENDER_WORDS, " ", t, flags=re.I)

SCRUBBED = [scrub(t) for t in TEXTS]
best_arch, best_rep = BEST.split("_", 1)
RESULTS_BY[f"{BEST}_scrubbed"] = run_system(best_rep, best_arch, texts=SCRUBBED, tag="_scrubbed")
display(pd.DataFrame([compare(BEST, f"{BEST}_scrubbed", m, r, plot=False)
                      for m, r in [(PRIMARY, ROPE), ("rms_gap", ROPE_GAP)]]).round(3))

In [ ]:
SWAP = {"he": "she", "she": "he", "his": "her", "her": "his", "him": "her",
        "himself": "herself", "herself": "himself", "mr": "ms", "ms": "mr", "mrs": "mr"}   # 'her' is ambiguous (his/him)
def swap(t):
    def rep(m):
        w = SWAP[m.group(0).lower()]
        return w.capitalize() if m.group(0)[0].isupper() else w
    return re.sub(r"\b(?:" + "|".join(SWAP) + r")\b", rep, t, flags=re.I)

tr, va = FOLDS[SEEDS[0]][0]
enc = REPRESENTATIONS["bow"]([TEXTS[i] for i in tr])
lr_model = part1.loglin.fit(pairs(tr), enc, save_to=None)
orig = np.asarray(part1.loglin.predict([TEXTS[i] for i in va], lr_model, enc))
swapd = np.asarray(part1.loglin.predict([swap(TEXTS[i]) for i in va], lr_model, enc))
print(f"flip rate under gender swap: {(orig != swapd).mean():.2%}")

In [ ]:
# top-weighted tokens per occupation (assumes lr_model is a sklearn LogisticRegression; check part1.loglin)
id2tok = {i: t for t, i in tokenizer.get_vocab().items()}
W = lr_model.coef_
for occ in ["nurse", "surgeon", "paralegal", "software_engineer"]:
    top = np.argsort(-W[NAMES.index(occ)])[:12]
    print(f"{occ:18s}", [id2tok[i] for i in top])
for w in ["▁he", "▁she", "▁his", "▁her"]:
    if w in tokenizer.get_vocab():
        col = W[:, tokenizer.get_vocab()[w]]
        print(w, "-> most favoured:", [NAMES[i] for i in np.argsort(-col)[:3]])

## 12 · Summary for the backup slides

| Slide | Content from this notebook |
|---|---|
| Data & metrics | §1 plots (class shares, % female), majority baseline, why macro-F1, ROPE choice |
| Features | §3 tokenizer choices + examples, §4 table of R1–R3 |
| Architectures | §5 table, hyperparameters, parameter counts |
| Results | §8 summary table + box plots, §9 posterior plots and P(A)/P(tie)/P(B)/effect size |
| Extra | §10 gap vs % female, gendered confusions, §11 scrubbing / swap test |

**Key findings (fill in):**
1. …
2. …
3. …

## 13 · AI-assistance & sources log

| Date | Tool | What was delegated | What I checked / changed |
|---|---|---|---|
| 2026-11-03 | Claude | Dataset research summary; notebook skeleton | … |

Sources: De-Arteaga et al. 2019 (Bias in Bios); Ravfogel et al. 2020 (HF version); Benavoli et al. 2017 (correlated t); NLP1 tutorial 1.